In [1]:
import os, sys, math, json, glob
import torch
from torch.utils.data import DataLoader, Dataset 
import numpy as np
import cv2
from torchvision import transforms


In [2]:

path = "./data/phys101/scenarios"
all_videos = glob.glob(f"{path}/**/*.mp4", recursive=True)
print("num videos: ", len(all_videos), "first 3: ", all_videos[:3])


num videos:  15190 first 3:  ['/Users/ashishneupane/data/phys101/scenarios/multi/w_block_05/01_02/03/Kinect_RGB_1.mp4', '/Users/ashishneupane/data/phys101/scenarios/multi/w_block_05/01_02/03/Kinect_RGB-D_1.mp4', '/Users/ashishneupane/data/phys101/scenarios/multi/w_block_05/01_02/03/Kinect_FullDepth_1.mp4']


In [10]:
os.listdir(f"{path}/ramp/w_pole_06")

['20_01', '10_01', '20_02', '10_02']

In [42]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, transform=None):
        self.video_paths = video_paths
        self.transform = transform or transforms.ToTensor()

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        cap = cv2.VideoCapture(video_path)
        frames = []
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            # Convert frame from BGR to RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Apply transformation
            frame = self.transform(frame)
            frames.append(frame)
        cap.release()
        # Stack frames into a single tensor
        video_tensor = torch.stack(frames)
        return video_tensor


In [43]:
# Define a transform if needed, e.g., resizing, normalization, etc.
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),  # Example resize, adjust to your needs
    transforms.ToTensor(),
    # Add more transformations if needed
])

# Create the dataset and dataloader
video_dataset = VideoDataset(all_videos, transform=transform)
video_dataloader = DataLoader(video_dataset, batch_size=1, shuffle=True)

# Iterate over the DataLoader
num_iter = 0
max_iter = 500

for batch_idx, batch_data in enumerate(video_dataloader):
    # batch_data shape: BatchSize x Frames x ColorChannels x FrameHeight x FrameWidth
    # batch_data is a batch of video tensors
    # Process your batch_data here (e.g., pass it through your neural network)
    #print(type(batch_idx), batch_idx, type(batch_data), batch_data.shape)
    num_iter += 1

    if num_iter >= max_iter:
        break